# STAC data prep — HydroLAKES (GeoParquet)

Converts the HydroLAKES polygon shapefile to a single **GeoParquet** file and writes the enriched metadata next to it.

- **Case:** vector data, one layer, 1.4 million lake/reservoir polygons with 21 attribute columns, EPSG:4326, no time dimension (~1.5 GB as shapefile).
- **Format:** one GeoParquet (see `DATA_FORMATS.md`, vector row; same approach as `14_groundwater_exploration` and `13_crop_productivity_correction`). Rows are sorted along a Hilbert curve and a bbox covering column is written, so readers can filter spatially without loading everything.
- **OGC service:** not created. The GeoServer of the Global Coastal Atlas (2.28.5) has no GeoParquet data store, and the shapefile is not published instead.
- **Metadata template:** `metadata/metadata_HydroLAKES.json`. It is incomplete (empty bbox, time, keywords, media type, collection id, wrong DOI), so the missing fields are filled below.

Source: `P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\HydroLAKES_polys_v10.shp`

**Running in docker:** the `P:` drive is not visible in the container, so run `docker\run_notebook_with_p_drive.ps1 -Notebook 18_hydrolakes.ipynb -Variable HydroLAKES_polys_v10` from the host. It copies the shapefile parts to a local mirror, executes this notebook, copies `stac\` back to `P:`, verifies the hashes and removes the mirror. The notebook needs about 6 GB of memory.

### Import packages

In [1]:
import json
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pyarrow.parquet as pq

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.

In [2]:
VARIABLE = "HydroLAKES_polys_v10"
COLLECTION_ID = "hydrolakes"

# DATA_VIVIAN is set in docker, where the P: drive is not mounted (a local mirror is used and stac/ is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_shp = data_vivian / f"{VARIABLE}.shp"
metadata_template_path = stac_repo / "metadata" / "metadata_HydroLAKES.json"

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
parquet_dir = processed_data_dir / "parquet"
parquet_dir.mkdir(parents=True, exist_ok=True)

for p in (source_shp, metadata_template_path):
    if not p.is_file():
        raise FileNotFoundError(p)
print("Source:", source_shp)
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

Source: /workspace/africa_data/Data_Vivian/HydroLAKES_polys_v10.shp
Metadata template: /workspace/global-coastal-atlas/STAC/data/metadata/metadata_HydroLAKES.json
Output dir: /workspace/africa_data/Data_Vivian/stac/hydrolakes


### Read raw data

Takes a few minutes and about 6 GB of memory.

In [3]:
gdf = gpd.read_file(source_shp, engine="pyogrio")

print("rows:", len(gdf), "| crs:", gdf.crs)
print("geometry types:", gdf.geom_type.value_counts().to_dict())
print("Hylak_id unique:", gdf["Hylak_id"].is_unique, "| null geometries:", int(gdf.geometry.isna().sum()))
print("total bounds:", gdf.total_bounds.tolist())
gdf.drop(columns="geometry").head()

rows: 1427688 | crs: EPSG:4326
geometry types: {'Polygon': 1427688}
Hylak_id unique: True | null geometries: 0


total bounds: [-179.9999999999999, -55.86513888888899, 179.99999999999903, 83.57595062255876]


,Hylak_id,Lake_name,Country,Continent,Poly_src,Lake_type,Grand_id,Lake_area,Shore_len,Shore_dev,...,Vol_res,Vol_src,Depth_avg,Dis_avg,Res_time,Elevation,Slope_100,Wshd_area,Pour_long,Pour_lat
0,1,Caspian Sea,Russia,Europe,SWBD,1,0,377001.91,15829.37,7.27,...,0.0,1,200.5,8110.642,107883.0,-29,-1.0,1404108.0,47.717708,45.591934
1,2,Great Bear,Canada,North America,CanVec,1,0,30450.64,5331.72,8.62,...,0.0,1,72.2,535.187,47577.7,145,-1.0,147665.4,-123.505546,65.138384
2,3,Great Slave,Canada,North America,CanVec,1,0,26734.29,9814.16,16.93,...,0.0,1,59.1,4350.692,4203.2,148,-1.0,995312.3,-117.617115,61.311727
3,4,Winnipeg,Canada,North America,CanVec,3,709,23923.04,4018.32,7.33,...,31790.0,1,11.9,2244.727,1464.3,215,-1.0,919611.5,-97.863542,53.696359
4,5,Superior,United States of America,North America,SWBD,1,0,81843.92,5248.36,5.18,...,0.0,1,146.7,2869.953,48410.3,179,-1.0,209219.5,-84.460547,46.468593


### Check the CRS

The shapefile is in WGS84 (`GCS_WGS_1984`), the CRS the STAC catalog uses. A missing `.prj` is set to EPSG:4326; any other CRS is reprojected.

In [4]:
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
print("CRS:", gdf.crs.to_string())

CRS: EPSG:4326


### Make the metadata complete (dataset dependent)

Fill the fields that are empty (or wrong) in the template. Column descriptions follow the HydroLAKES technical documentation (v1.0, `HydroLAKES_TechDoc_v10.pdf`); please double-check the units that are marked with `?` before publishing.

In [5]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

# Descriptions per attribute column (used as table:columns in the STAC item)
table_columns = {
    "Hylak_id": "Unique lake identifier (shared with HydroRIVERS / HydroBASINS)",
    "Lake_name": "Lake or reservoir name",
    "Country": "Country in which the lake is located",
    "Continent": "Continent in which the lake is located",
    "Poly_src": "Source of the lake polygon (e.g. SWBD, CanVec, CLAKES)",
    "Lake_type": "Lake type: 1 = lake, 2 = reservoir, 3 = lake control structure",
    "Grand_id": "ID of the reservoir in the GRanD database (0 if not a GRanD reservoir)",
    "Lake_area": "Lake surface area (km2)",
    "Shore_len": "Shoreline length (km)",
    "Shore_dev": "Shoreline development: shoreline length / circumference of a circle with the same area (-)",
    "Vol_total": "Total lake or reservoir volume (million m3)",
    "Vol_res": "Reported reservoir volume (million m3)",
    "Vol_src": "Source of the volume: 1 = reported total volume, 2 = reported reservoir volume, 3 = estimated",
    "Depth_avg": "Average lake depth (m)",
    "Dis_avg": "Average long-term discharge through the lake (m3/s)",
    "Res_time": "Average residence time (days)",
    "Elevation": "Elevation of the lake surface (m above sea level)",
    "Slope_100": "Average slope within a 100 m buffer around the lake (degrees ?)",
    "Wshd_area": "Area of the upstream watershed (km2)",
    "Pour_long": "Longitude of the lake pour point (degrees)",
    "Pour_lat": "Latitude of the lake pour point (degrees)",
    "geometry": "Lake / reservoir shoreline polygon (EPSG:4326)",
}
missing = set(gdf.columns) - set(table_columns)
assert not missing, f"Columns without description: {missing}"

west, south, east, north = (float(v) for v in gdf.total_bounds)
bbox = [max(west, -180.0), max(south, -90.0), min(east, 180.0), min(north, 90.0)]

# Extra info that the raw template does not contain
metadata.update({
    "TITLE_ABBREVIATION": "HydroLAKES",
    "HISTORY": [
        "HydroLAKES version 1.0 (Messager et al., 2016), shapefile HydroLAKES_polys_v10.shp",
        "Converted to a single GeoParquet (zstd, EPSG:4326, rows sorted along a Hilbert curve, bbox covering column)",
        "Geometries and attributes are unchanged",
    ],
    "MEDIA_TYPE": "application/vnd.apache.parquet",
    "DATA_MODEL": "vector_geoparquet",
    "DIMENSIONS": [],
    "SPATIAL_EXTENT": [round(v, 6) for v in bbox],
    # Publication year of HydroLAKES v1.0; the dataset has no time dimension
    "TEMPORAL_EXTENT": ["2016-01-01T00:00:00Z", ""],
    "KEYWORDS": ["lakes", "reservoirs", "surface water", "hydrology", "freshwater", "HydroSHEDS", "global"],
    "TAGS": ["lakes", "reservoirs", "polygons", "geoparquet"],
    "LONG_NAME": "Global lake and reservoir shoreline polygons",
    "UNITS": "",
    "COMMENT": (
        "Vector layer with one polygon per lake or reservoir (surface area >= 10 ha). "
        "Units differ per attribute and are listed in TABLE_COLUMNS. "
        "The temporal extent is the publication year; the layer is static."
    ),
    "AUTHOR": "Messager ML, Lehner B, Grill G, Nedeva I, Schmitt O",
    "CITATION": (
        "Messager ML, Lehner B, Grill G, Nedeva I, Schmitt O (2016) Estimating the volume and age of water stored "
        "in global lakes using a geo-statistical approach. Nature Communications 7:13603. "
        "https://doi.org/10.1038/ncomms13603"
    ),
    "DOI": "https://doi.org/10.1038/ncomms13603",
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": None,
    "NODATA": None,
    "DATA_TYPE": "GeoParquet",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": "",  # no OGC service: GeoServer has no GeoParquet data store
    "FEATURE_COUNT": int(len(gdf)),
    "TABLE_COLUMNS": table_columns,
})

print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("FEATURE_COUNT:", metadata["FEATURE_COUNT"])

SPATIAL_EXTENT: [-180.0, -55.865139, 180.0, 83.575951]
FEATURE_COUNT: 1427688


### Write data to GeoParquet

`zstd` compression, row groups of 100 000 polygons, rows ordered along a Hilbert curve so that nearby lakes share a row group, and a `bbox` covering column for spatial filtering.

In [6]:
gdf = gdf.iloc[gdf.geometry.hilbert_distance().argsort()]

out_path = parquet_dir / f"{COLLECTION_ID}.parquet"
gdf.to_parquet(
    out_path,
    index=False,
    compression="zstd",
    row_group_size=100_000,
    write_covering_bbox=True,
)

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

before_mb = sum(p.stat().st_size for p in source_shp.parent.glob(f"{VARIABLE}.*")) / 1e6
print(f"Wrote GeoParquet: {out_path} ({out_path.stat().st_size / 1e6:.0f} MB, shapefile parts {before_mb:.0f} MB)")
print("Wrote metadata:", metadata_output_path)

Wrote GeoParquet: /workspace/africa_data/Data_Vivian/stac/hydrolakes/parquet/hydrolakes.parquet (1011 MB, shapefile parts 1513 MB)
Wrote metadata: /workspace/africa_data/Data_Vivian/stac/hydrolakes/metadata_hydrolakes.json


### Verify the output

Same rows, columns and values as the shapefile, EPSG:4326, readable as GeoParquet.

In [7]:
meta = pq.ParquetFile(out_path).metadata
print("row groups:", meta.num_row_groups, "| rows:", meta.num_rows)
assert meta.num_rows == len(gdf)

new = gpd.read_parquet(out_path).set_index("Hylak_id").sort_index()
old = gdf.set_index("Hylak_id").sort_index()
assert new.crs.to_epsg() == 4326
assert new.index.is_unique and new.index.equals(old.index)
assert list(new.columns.drop("bbox", errors="ignore")) == list(old.columns)

attrs = [c for c in old.columns if c != "geometry"]
pd.testing.assert_frame_equal(pd.DataFrame(new[attrs]), pd.DataFrame(old[attrs]))
assert new.geometry.geom_equals_exact(old.geometry, tolerance=0).all(), "Geometries changed"
print("OK: hydrolakes GeoParquet and metadata are ready for the STAC builder notebook.")

row groups: 15 | rows: 1427688


OK: hydrolakes GeoParquet and metadata are ready for the STAC builder notebook.
